# ECE 453 — Homework 1, Problem 4 (`numpy` Exercises) — SOLUTION

Fill in the two functions below.

**Rules**
- Do **not** use `for` / `while` loops or list comprehensions anywhere in your answers.
- Watch the array shapes. In `numpy` a shape `(N,)` array and a shape `(N, 1)` array do
  *not* behave the same way, and mixing them up tends to produce bugs that fail silently
  rather than raising an error.
- Each cell marked `# --- check ... ---` is there to help you. It uses a slow reference
  implementation that you may **not** use in your own answer.

When you are done, run every cell top to bottom, export the notebook to PDF, and attach
the PDF to your Gradescope write-up.

In [ ]:
import numpy as np

rng = np.random.default_rng(453)

## (a) Row-wise $\ell_1$ norms — 10 points

Given a matrix $A \in \mathbb{R}^{N \times M}$ (a shape `(N, M)` array), return a vector
$x \in \mathbb{R}^N$ (a shape `(N,)` array) such that $x_i$ is the $\ell_1$ norm of the
$i$-th row of $A$.

Your solution should be a **single line**.

In [ ]:
def row_l1_norms(A):
    """
    A       : np.ndarray, shape (N, M)
    returns : np.ndarray, shape (N,) -- the l1 norm of each row of A
    """
    return np.abs(A).sum(axis=1)

In [ ]:
# --- check (a) ---
A_test = rng.normal(size=(5, 4))

def _ref_row_l1(A):            # slow reference, for checking only -- do not use in your answer
    return np.array([np.abs(A[i]).sum() for i in range(A.shape[0])])

out = row_l1_norms(A_test)
assert out.shape == (5,), f"expected shape (5,), got {out.shape}"
assert np.allclose(out, _ref_row_l1(A_test)), "values do not match"
print("(a) looks correct")

## (b) Closest cell tower — 10 points

`A` is a shape `(M, D)` array holding the locations of $M$ cell towers in $D$-dimensional
space, and `B` is a shape `(N, D)` array holding the locations of $N$ home addresses.
Return a shape `(N,)` array `x` where `x[i]` is the index of the tower closest (in
Euclidean distance) to home `i`. For example, if `B[8, :]` is closest to `A[6, :]`, then
`x[8] == 6`.

**A buggy implementation is provided below — your job is to fix it.** It runs without
raising an exception, but it does not compute what it claims to. Maximum 4 lines of code
(the `def` and docstring lines do not count); no partial credit if you use more.

Hint: $\|x - y\|_2^2 = \|x\|_2^2 + \|y\|_2^2 - 2 x^\top y$.

In [ ]:
def closest_tower(A, B):
    """
    A       : np.ndarray, shape (M, D) -- tower locations
    B       : np.ndarray, shape (N, D) -- home locations
    returns : np.ndarray, shape (N,)   -- for each home, the index of the nearest tower
    """
    sq_dists = (B**2).sum(axis=1)[:, None] + (A**2).sum(axis=1)[None, :] - 2 * B @ A.T
    return np.argmin(sq_dists, axis=1)

In [ ]:
# --- check (b) ---
A_towers = rng.normal(size=(7, 3))     # M = 7 towers,  D = 3
B_homes  = rng.normal(size=(11, 3))    # N = 11 homes

def _ref_closest(A, B):        # slow reference, for checking only -- do not use in your answer
    return np.array([int(np.argmin(((B[i] - A) ** 2).sum(axis=1))) for i in range(B.shape[0])])

out = closest_tower(A_towers, B_homes)
assert out.shape == (B_homes.shape[0],), f"expected shape {(B_homes.shape[0],)}, got {out.shape}"
assert np.array_equal(out, _ref_closest(A_towers, B_homes)), "indices are not the nearest towers"
print("(b) looks correct")

### What was wrong with the provided code

Two independent bugs, neither of which raises an exception:

1. **Sign.** The identity is $\|b - a\|_2^2 = \|b\|_2^2 + \|a\|_2^2 - 2\,b^\top a$, so the
   cross term must be subtracted. With `+ 2 * B @ A.T` the code computes
   $\|b + a\|_2^2$ instead, and picks the tower nearest to $-b$.
2. **Axis.** `sq_dists` has shape `(N, M)`: rows index homes, columns index towers. The
   minimum must therefore be taken over the columns, `axis=1`. Taking `axis=0` minimises
   over homes and returns a shape `(M,)` array — the wrong length, which is the quickest
   way to notice the bug.

---

Once both checks print `looks correct`, restart the kernel and run all cells in order, then
export to PDF and attach it to your Gradescope submission.